# Clustering sensitivity

Validation check V3. ADR 0002 accepts that results are sensitive to the DBSCAN radius; accepting that obliges quantifying it. The question is not whether the clusters change — they will — but whether the **technology mix** changes with them.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from kajiado_lceo import load_config
from kajiado_lceo.pipeline import run

config = load_config("../config", scenario="baseline")
config.scenario

In [ ]:
import numpy as np

from kajiado_lceo.sample import generate_sample_county

layers = generate_sample_county(seed=42)

## Sweep eps x MinPts

The proposal's stated ranges: eps 50-100 m, MinPts 3-5.

In [ ]:
rows = []
for eps in config.get("clustering.eps_sweep_m"):
    for min_pts in config.get("clustering.min_pts_sweep"):
        variant = (config
                   .with_override_path("clustering.eps_m", eps)
                   .with_override_path("clustering.min_pts", min_pts))
        result = run(variant, inputs=layers, write_outputs=False)
        mix = result.summary["technology_mix"]
        rows.append({"eps_m": eps, "min_pts": min_pts,
                     "settlements": result.summary["totals"]["settlements"],
                     "mean_lcoe": result.summary["totals"]["mean_lcoe_usd_per_kwh"],
                     **{f"n_{k}": v for k, v in mix.items()}})

sweep = pd.DataFrame(rows).fillna(0)
sweep

## Does the recommendation move?

Report the **share** of settlements on each technology, not the count: the count changes mechanically with eps, the share is the substantive question.

In [ ]:
tech_cols = [c for c in sweep.columns if c.startswith("n_")]
shares = sweep[tech_cols].div(sweep[tech_cols].sum(axis=1), axis=0)
shares.index = sweep.apply(lambda r: f"eps={r.eps_m:.0f}, k={r.min_pts:.0f}", axis=1)
shares.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.ylabel("Share of settlements"); plt.show()